In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

filepath = 'Waveforms/DATASET_02/TEST4_20_JULY_1536.csv'

# Peek at the structure
with open(filepath, 'r') as f:
    for _ in range(10):
        print(f.readline().strip())

Time,Channel A
(s),(V)

0.00000000,-1.00570691
0.00005000,-1.00570691
0.00010000,-1.00570691
0.00015000,-1.00570691
0.00020000,-1.00570691
0.00025000,-1.00570691
0.00030000,-1.02401805


In [2]:
# Load the data, skipping the unit row
df = pd.read_csv(filepath, skiprows=[1])

# Plot the full waveform to find the trigger location
plt.figure(figsize=(10, 4))
plt.plot(df['Time'], df['Channel A'])
plt.title("Full Waveform - TEST4")
plt.xlabel("Time (s)")
plt.ylabel("Voltage (V)")
plt.grid(True)
plt.savefig('full_waveform_test4.png')
plt.close()

print(f"Total rows: {len(df)}")
print(f"Time range: {df['Time'].min()} to {df['Time'].max()}")

Total rows: 96764
Time range: 0.0 to 4.83814988


In [3]:
# There are two triggers! One around t=1.25s and another around t=2.5s.
# Let's inspect the second trigger since it seems to have a long, undisturbed baseline settling afterwards.
# Let's look at the baseline from t=0 to 1.0 to check the default level.
baseline_true = np.mean(df[(df['Time'] >= 0.1) & (df['Time'] <= 1.0)]['Channel A'].values)
print("Initial baseline:", baseline_true)

# Let's check the baseline after the second event (t=3.5 to 4.5)
baseline_post = np.mean(df[(df['Time'] >= 3.5) & (df['Time'] <= 4.5)]['Channel A'].values)
print("Post-second trigger baseline:", baseline_post)

# Let's perform FFT on the second decay region (e.g. from t=2.55 to 2.85)
df_fft = df[(df['Time'] >= 2.55) & (df['Time'] <= 2.85)].copy()
dt = df_fft['Time'].iloc[1] - df_fft['Time'].iloc[0]
fs = 1.0 / dt
print("Sampling frequency:", fs)

y = df_fft['Channel A'].values - baseline_post
n = len(y)
yf = np.fft.rfft(y)
xf = np.fft.rfftfreq(n, d=dt)

peak_idx = np.argmax(np.abs(yf))
fd = xf[peak_idx]
print(f"Damped frequency fd: {fd} Hz")

# Let's do the same for the first decay region (t=1.30 to 1.60) just to see if they match
df_fft1 = df[(df['Time'] >= 1.30) & (df['Time'] <= 1.60)].copy() 
y1 = df_fft1['Channel A'].values - np.mean(df_fft1['Channel A'].values)
yf1 = np.fft.rfft(y1)
xf1 = np.fft.rfftfreq(len(y1), d=dt)
print(f"First event frequency fd: {xf1[np.argmax(np.abs(yf1))]} Hz")

Initial baseline: -1.0101411037975667
Post-second trigger baseline: -0.9334101952025001
Sampling frequency: 20000.000000046613
Damped frequency fd: 473.3333333344365 Hz
First event frequency fd: 473.3333333344365 Hz
